In [1]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import os
import re
import time

import pandas as pd
import requests
from tqdm import tqdm

BASE_URL = "https://www.truecar.com"
DATA_DIR = "./data"

DEFAULT_HEADERS = {
    "User-Agent": os.getenv(
        "TRUECAR_USER_AGENT",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": os.getenv("TRUECAR_ACCEPT_LANGUAGE", "en-US,en;q=0.9"),
    "Accept-Encoding": "gzip, deflate",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Sec-Fetch-User": "?1",
    "Upgrade-Insecure-Requests": "1",
}



In [2]:
def request_headers(referer=None):
    headers = dict(DEFAULT_HEADERS)
    cookie = os.getenv("TRUECAR_COOKIE")
    if cookie:
        headers["Cookie"] = cookie
    if os.getenv("TRUECAR_SEC_CH_UA"):
        headers["sec-ch-ua"] = os.environ["TRUECAR_SEC_CH_UA"]
    if os.getenv("TRUECAR_SEC_CH_UA_MOBILE"):
        headers["sec-ch-ua-mobile"] = os.environ["TRUECAR_SEC_CH_UA_MOBILE"]
    if os.getenv("TRUECAR_SEC_CH_UA_PLATFORM"):
        headers["sec-ch-ua-platform"] = os.environ["TRUECAR_SEC_CH_UA_PLATFORM"]

    referer = os.getenv("TRUECAR_REFERER") or referer
    if referer:
        headers["Referer"] = referer
    return headers


def get_soup(url, delay_seconds=0.75, referer=None):
    time.sleep(delay_seconds)
    response = requests.get(url, headers=request_headers(referer=referer), timeout=30)
    if response.status_code == 403:
        print(f"Blocked with HTTP 403 for {url}")
        print("Use a legitimate TRUECAR_COOKIE / TRUECAR_USER_AGENT from your browser session, reduce volume, or use a permitted data source.")
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def slug_city(city):
    return city.strip().lower().replace(" ", "-")


def location_key(city, state):
    return f"{slug_city(city)}_{state.strip().lower()}"


def truecar_listing_url(city, state, page=1, page_size=100, search_radius=250, exclude_expanded_delivery=True):
    expanded_delivery = "true" if exclude_expanded_delivery else "false"
    return (
        f"{BASE_URL}/used-cars-for-sale/listings/location-{slug_city(city)}-{state.strip().lower()}/"
        f"?stock_type=used&page_size={page_size}&searchRadius={search_radius}"
        f"&excludeExpandedDelivery={expanded_delivery}&page={page}"
    )



In [3]:
def first_text(parent, selector):
    element = parent.select_one(selector) if parent else None
    return element.get_text(separator=" ", strip=True) if element else None


def card_position(card, fallback):
    ordered_parent = card.find_parent("li", style=re.compile(r"order\s*:"))
    if ordered_parent:
        match = re.search(r"order\s*:\s*(\d+)", ordered_parent.get("style", ""))
        if match:
            return int(match.group(1))
    return fallback


def listing_card_containers(soup):
    cards = soup.select('[data-test="vehicleListingCard"]')
    if cards:
        return cards
    containers = []
    for link in soup.select('a[data-test="cardLinkCover"]'):
        container = link.find_parent(lambda tag: tag.name in {"li", "div"} and tag.select_one('a[data-test="cardLinkCover"]'))
        if container:
            containers.append(container)
    return containers


def vin_from_card(card, link_tag):
    listing = card.select_one('[data-test="usedListing"]')
    if listing and listing.get("data-test-item"):
        return listing.get("data-test-item")
    href = link_tag.get("href", "") if link_tag else ""
    match = re.search(r"/listing/([^/]+)/", href)
    return match.group(1) if match else None


def image_from_card(card):
    image = card.select_one('[data-test="listingCardImage"]')
    if not image:
        return None
    for attribute in ("src", "data-src"):
        value = image.get(attribute)
        if value:
            return urljoin(BASE_URL, value)
    srcset = image.get("srcset")
    if srcset:
        first_source = srcset.split(",", 1)[0].strip().split(" ", 1)[0]
        if first_source:
            return urljoin(BASE_URL, first_source)
    return None


def parse_listing_card(card, city, state, page, fallback_position):
    link_tag = card.select_one('a[data-test="cardLinkCover"]')
    relative_link = link_tag.get("href") if link_tag else None
    if not relative_link:
        return None

    row = {
        "source_city": city,
        "source_state": state.upper(),
        "source_metro": location_key(city, state),
        "source_page": page,
        "source_position": card_position(card, fallback_position),
        "url": urljoin(BASE_URL, relative_link),
        "vin": vin_from_card(card, link_tag),
        "listing_image_url": image_from_card(card),
        "title": first_text(card, '[data-test="vehicleCardInfo"]'),
        "mileage_listed": first_text(card, '[data-test="vehicleMileage"]'),
        "list_price_displayed": first_text(card, '[data-test="vehicleCardPricingPrice"]'),
        "dealer_info": first_text(card, '[data-test="vehicleCardFooter"]') or first_text(card, '[data-test="vehicleCardDeliveryDetails"]'),
    }
    return row



In [4]:
def collect_listing_cards(city, state, max_pages=1, page_size=100, search_radius=250, delay_seconds=0.75):
    rows = []
    for page in tqdm(range(1, max_pages + 1), desc=f"{location_key(city, state)} listing pages"):
        url = truecar_listing_url(
            city=city,
            state=state,
            page=page,
            page_size=page_size,
            search_radius=search_radius,
        )
        try:
            soup = get_soup(url, delay_seconds=delay_seconds, referer=BASE_URL)
        except requests.RequestException as exc:
            print(f"Request failed for {city}, {state}, page {page}: {exc}")
            continue

        cards = listing_card_containers(soup)
        if not cards:
            print(f"No listing cards found for {city}, {state}, page {page}.")
            break

        for fallback_position, card in enumerate(cards, start=1):
            row = parse_listing_card(card, city, state, page, fallback_position)
            if row:
                rows.append(row)
    return rows


def save_listing_cards(rows, city, state, data_dir=DATA_DIR):
    os.makedirs(data_dir, exist_ok=True)
    key = location_key(city, state)
    json_path = os.path.join(data_dir, f"truecar_links_{key}.json")
    csv_path = os.path.join(data_dir, f"truecar_links_{key}.csv")

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    pd.DataFrame(rows).to_csv(csv_path, index=False)

    print(f"Saved {len(rows)} listing cards for {city.title()}, {state.upper()}")
    print(f"- {json_path}")
    print(f"- {csv_path}")
    return json_path, csv_path



In [5]:
def extract_car_details(entry, delay_seconds=0.75):
    car_data = entry.copy()
    link = entry["url"]

    try:
        soup = get_soup(link, delay_seconds=delay_seconds, referer=entry.get("return_to") or BASE_URL)
    except requests.RequestException as exc:
        car_data["scrape_error"] = str(exc)
        return car_data

    try:
        container = soup.select_one("div.row.pt-3")
        if container:
            details = container.select("div.flex.items-center")
            for detail in details:
                text = detail.get_text(separator=" ", strip=True)
                if ":" in text:
                    key, value = text.split(":", 1)
                    car_data[key.strip().lower().replace(" ", "_")] = value.strip()
                elif "VIN" in text:
                    car_data["vin"] = text.split("VIN:", 1)[-1].strip()
                elif "Stock Number" in text:
                    car_data["stock_number"] = text.split("Stock Number:", 1)[-1].strip()
                elif "miles" in text:
                    car_data["mileage"] = text.strip()
                elif "Listed" in text:
                    car_data["listed_since"] = text.strip()

        for heading, output_name in [
            ("Options & packages", "options_and_packages"),
            ("Popular features", "popular_features"),
            ("Standard features", "standard_features"),
        ]:
            section = soup.find("h2", string=heading)
            if section:
                items = [
                    item.get_text(separator=" ", strip=True)
                    for item in section.find_next("div").find_all("div", class_="flex items-center")
                ]
                car_data[output_name] = ", ".join(item for item in items if item)

        price_section = soup.find("div", {"id": "usedPriceGraph"})
        if price_section:
            for item in price_section.select('div[data-test="usedListingPriceGraphLineItem"]'):
                label = item.get("data-test-item")
                text = item.get_text(separator="|", strip=True)
                if label and "|" in text:
                    _, value = text.split("|", 1)
                    car_data[label.lower().replace(" ", "_")] = value.strip()

            description = price_section.find("div", {"data-test": "usedListingPriceGraphDescription"})
            if description:
                car_data["price_description"] = description.get_text(separator=" ", strip=True)

        seller_notes_header = soup.find("h2", string="Seller Notes")
        if seller_notes_header:
            seller_div = seller_notes_header.find_next("div", class_="see-more")
            if seller_div:
                car_data["seller_notes"] = seller_div.get_text(separator=" ", strip=True)
    except Exception as exc:
        car_data["scrape_error"] = str(exc)

    return car_data



In [6]:
def clean_detail_dataframe(results):
    df = pd.DataFrame(results)
    if df.empty:
        return df

    def split_title(title):
        if not isinstance(title, str):
            return [None, None, None, None]
        parts = title.replace("Used", "").strip().split()
        year = parts[0] if len(parts) > 0 else None
        make = parts[1] if len(parts) > 1 else None
        model = parts[2] if len(parts) > 2 else None
        trim = " ".join(parts[3:]) if len(parts) > 3 else None
        return [year, make, model, trim]

    if "title" in df.columns:
        df[["year", "make", "model", "trim"]] = df["title"].apply(lambda value: pd.Series(split_title(value)))

    if "dealer_info" in df.columns:
        df[["dealer_city_state", "dealer_distance"]] = df["dealer_info"].str.extract(r"(.+?,\s*[A-Z]{2})\s*\((.*?)\)", expand=True)
        df[["dealer_city", "dealer_state"]] = df["dealer_city_state"].str.extract(r"(.+),\s*([A-Z]{2})")

    if "price_description" in df.columns:
        df["price_description"] = df["price_description"].str.replace(r"([a-zA-Z])(\$)", r" ", regex=True)
    if "seller_notes" in df.columns:
        df["seller_notes"] = df["seller_notes"].str.replace(r"\.(\w)", r". ", regex=True)
    return df


def scrape_all_car_details(json_path, output_json, output_csv, delay_seconds=0.75):
    with open(json_path, "r", encoding="utf-8") as f:
        car_links = json.load(f)

    results = []
    for entry in tqdm(car_links, desc="Scraping car details"):
        results.append(extract_car_details(entry, delay_seconds=delay_seconds))

    df = clean_detail_dataframe(results)
    df.to_csv(output_csv, index=False)
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print("Cleaned and saved car details to:")
    print(f"- {output_csv}")
    print(f"- {output_json}")
    return df



In [7]:
def scrape_city_state(city, state, max_pages=1, page_size=100, search_radius=250, scrape_details=False, delay_seconds=0.75):
    key = location_key(city, state)
    rows = collect_listing_cards(
        city=city,
        state=state,
        max_pages=max_pages,
        page_size=page_size,
        search_radius=search_radius,
        delay_seconds=delay_seconds,
    )
    json_path, csv_path = save_listing_cards(rows, city, state)

    detail_df = None
    if scrape_details and rows:
        output_json = f"{DATA_DIR}/truecar_details_{key}.json"
        output_csv = f"{DATA_DIR}/truecar_details_{key}.csv"
        detail_df = scrape_all_car_details(json_path, output_json, output_csv, delay_seconds=delay_seconds)

    return {
        "city": city,
        "state": state.upper(),
        "source_metro": key,
        "listing_count": len(rows),
        "links_json": json_path,
        "links_csv": csv_path,
        "detail_rows": 0 if detail_df is None else len(detail_df),
    }


def scrape_locations(locations, max_pages=1, page_size=100, search_radius=250, scrape_details=False, delay_seconds=0.75):
    summaries = []
    for city, state in locations:
        summaries.append(
            scrape_city_state(
                city=city,
                state=state,
                max_pages=max_pages,
                page_size=page_size,
                search_radius=search_radius,
                scrape_details=scrape_details,
                delay_seconds=delay_seconds,
            )
        )
    summary_df = pd.DataFrame(summaries)
    os.makedirs(DATA_DIR, exist_ok=True)
    summary_path = f"{DATA_DIR}/truecar_boston_metro_download_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved run summary to {summary_path}")
    return summary_df



In [8]:
BOSTON_METRO_LOCATIONS = [
    ("Boston", "MA"),
    ("Worcester", "MA"),
    ("Providence", "RI"),
    ("Manchester", "NH"),
    ("Nashua", "NH"),
    ("Lowell", "MA"),
    ("Hartford", "CT"),
    ("Springfield", "MA"),
]

# First trial: one listing page per location, no detail-page fan-out yet.
# After the listing-card download works, set SCRAPE_DETAILS = True to enrich every car from each first page.
MAX_PAGES = 1
PAGE_SIZE = 100
SEARCH_RADIUS = 250
SCRAPE_DETAILS = False
DELAY_SECONDS = 0.75

summary = scrape_locations(
    BOSTON_METRO_LOCATIONS,
    max_pages=MAX_PAGES,
    page_size=PAGE_SIZE,
    search_radius=SEARCH_RADIUS,
    scrape_details=SCRAPE_DETAILS,
    delay_seconds=DELAY_SECONDS,
)
summary



boston_ma listing pages: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]


Blocked with HTTP 403 for https://www.truecar.com/used-cars-for-sale/listings/location-boston-ma/?stock_type=used&page_size=100&searchRadius=250&excludeExpandedDelivery=true&page=1
Use a legitimate TRUECAR_COOKIE / TRUECAR_USER_AGENT from your browser session, reduce volume, or use a permitted data source.
Request failed for Boston, MA, page 1: 403 Client Error: Forbidden for url: https://www.truecar.com/used-cars-for-sale/listings/location-boston-ma/?stock_type=used&page_size=100&searchRadius=250&excludeExpandedDelivery=true&page=1
Saved 0 listing cards for Boston, MA
- ./data\truecar_links_boston_ma.json
- ./data\truecar_links_boston_ma.csv


worcester_ma listing pages: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]


Blocked with HTTP 403 for https://www.truecar.com/used-cars-for-sale/listings/location-worcester-ma/?stock_type=used&page_size=100&searchRadius=250&excludeExpandedDelivery=true&page=1
Use a legitimate TRUECAR_COOKIE / TRUECAR_USER_AGENT from your browser session, reduce volume, or use a permitted data source.
Request failed for Worcester, MA, page 1: 403 Client Error: Forbidden for url: https://www.truecar.com/used-cars-for-sale/listings/location-worcester-ma/?stock_type=used&page_size=100&searchRadius=250&excludeExpandedDelivery=true&page=1
Saved 0 listing cards for Worcester, MA
- ./data\truecar_links_worcester_ma.json
- ./data\truecar_links_worcester_ma.csv


providence_ri listing pages: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]


...

hartford_ct listing pages: 100%|██████████| 1/1 [00:01<00:00,  1.23s/it]


Blocked with HTTP 403 for https://www.truecar.com/used-cars-for-sale/listings/location-hartford-ct/?stock_type=used&page_size=100&searchRadius=250&excludeExpandedDelivery=true&page=1
Use a legitimate TRUECAR_COOKIE / TRUECAR_USER_AGENT from your browser session, reduce volume, or use a permitted data source.
Request failed for Hartford, CT, page 1: 403 Client Error: Forbidden for url: https://www.truecar.com/used-cars-for-sale/listings/location-hartford-ct/?stock_type=used&page_size=100&searchRadius=250&excludeExpandedDelivery=true&page=1
Saved 0 listing cards for Hartford, CT
- ./data\truecar_links_hartford_ct.json
- ./data\truecar_links_hartford_ct.csv


springfield_ma listing pages: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Blocked with HTTP 403 for https://www.truecar.com/used-cars-for-sale/listings/location-springfield-ma/?stock_type=used&page_size=100&searchRadius=250&excludeExpandedDelivery=true&page=1
Use a legitimate TRUECAR_COOKIE / TRUECAR_USER_AGENT from your browser session, reduce volume, or use a permitted data source.
Request failed for Springfield, MA, page 1: 403 Client Error: Forbidden for url: https://www.truecar.com/used-cars-for-sale/listings/location-springfield-ma/?stock_type=used&page_size=100&searchRadius=250&excludeExpandedDelivery=true&page=1
Saved 0 listing cards for Springfield, MA
- ./data\truecar_links_springfield_ma.json
- ./data\truecar_links_springfield_ma.csv
Saved run summary to ./data/truecar_boston_metro_download_summary.csv


,city,state,source_metro,listing_count,links_json,links_csv,detail_rows
0,Boston,MA,boston_ma,0,./data\truecar_links_boston_ma.json,./data\truecar_links_boston_ma.csv,0
1,Worcester,MA,worcester_ma,0,./data\truecar_links_worcester_ma.json,./data\truecar_links_worcester_ma.csv,0
2,Providence,RI,providence_ri,0,./data\truecar_links_providence_ri.json,./data\truecar_links_providence_ri.csv,0
3,Manchester,NH,manchester_nh,0,./data\truecar_links_manchester_nh.json,./data\truecar_links_manchester_nh.csv,0
4,Nashua,NH,nashua_nh,0,./data\truecar_links_nashua_nh.json,./data\truecar_links_nashua_nh.csv,0
5,Lowell,MA,lowell_ma,0,./data\truecar_links_lowell_ma.json,./data\truecar_links_lowell_ma.csv,0
6,Hartford,CT,hartford_ct,0,./data\truecar_links_hartford_ct.json,./data\truecar_links_hartford_ct.csv,0
7,Springfield,MA,springfield_ma,0,./data\truecar_links_springfield_ma.json,./data\truecar_links_springfield_ma.csv,0
